In [ ]:
import pyaudio
import wave

w = wave.open("/usr/share/sounds/alsa/Side_Left.wav", "rb")
data = w.readframes(w.getnframes())
w.close()

p = pyaudio.PyAudio()
stream = p.open(format=p.get_format_from_width(2), 
                channels=1, 
                rate=48000, 
                output=True)

stream.write(data)

stream.stop_stream()
stream.close()

In [ ]:
import pyaudio
import wave

w = wave.open("/usr/share/sounds/alsa/Side_Left.wav", "rb")
data = w.readframes(w.getnframes())
w.close()

p = pyaudio.PyAudio()
stream = p.open(format=p.get_format_from_width(w.getsampwidth()), 
                channels=w.getnchannels(), 
                rate=w.getframerate(), 
                output=True)

stream.write(data)

stream.stop_stream()
stream.close()

In [ ]:
import pyaudio
import wave

w = wave.open("/usr/share/sounds/alsa/Side_Left.wav", "rb")
data = w.readframes(1024)

p = pyaudio.PyAudio()
stream = p.open(format=p.get_format_from_width(w.getsampwidth()), 
                channels=w.getnchannels(), 
                rate=w.getframerate(), 
                output=True)

while len(data) > 0:
    stream.write(data)
    data = w.readframes(1024)
    
stream.stop_stream()
stream.close()

In [ ]:
import pyaudio
import wave
import time

w = wave.open("/usr/share/sounds/alsa/Side_Left.wav", "rb")
p = pyaudio.PyAudio()

def callback(in_data, frame_count, time_info, status):
    data = w.readframes(frame_count)
    return (data, pyaudio.paContinue)

stream = p.open(format=p.get_format_from_width(w.getsampwidth()),
                channels=w.getnchannels(),
                rate=w.getframerate(),
                output=True,
                stream_callback=callback)

stream.start_stream()

while stream.is_active():
    print("main work...")
    time.sleep(0.1)

stream.stop_stream()
stream.close()
p.terminate()

- 코드를 중지하려면 메뉴에서 Kernel > Restart Kernel 클릭   

In [ ]:
import pyaudio
import wave
import time

w = wave.open("/usr/share/sounds/alsa/Side_Left.wav", "rb")
p = pyaudio.PyAudio()

def callback(in_data, frame_count, time_info, status):
    data = w.readframes(frame_count)
    mod = frame_count - len(data) // w.getsampwidth()
    if mod != 0:
        w.rewind()
        data += w.readframes(mod)

    return (data, pyaudio.paContinue)

stream = p.open(format=p.get_format_from_width(w.getsampwidth()),
                channels=w.getnchannels(),
                rate=w.getframerate(),
                output=True,
                stream_callback=callback)

stream.start_stream()

print("main work..", end ="")
while stream.is_active():
    print(".", end="")
    time.sleep(0.1)

stream.stop_stream()
stream.close()
p.terminate()

In [ ]:
import pyaudio
import numpy as np

volume = 0.5
fs = 48000
duration = 1.0  
f = 440.0

data = (np.sin(2 * np.pi * np.arange(fs * duration) * f/fs)).astype(np. float32)

p = pyaudio.PyAudio()
stream = p.open(format=pyaudio.paFloat32, channels=1, rate=fs, output=True)
stream.write(volume * data)

stream.stop_stream()
stream.close() 
p.terminate()

In [ ]:
import pyaudio
import numpy as np

class Tone:
    def __init__(self, volume=.5, rate=48000, channels=1): 
        self.volume = volume
        self.rate = rate
        self.channels = channels
        self.p = pyaudio.PyAudio()
        self.stream = self.p.open(format=pyaudio.paFloat32, channels=self.channels, rate=self.rate, output=True)

    def play(self, octave, note, duration):
        f = 2**(octave) * 55 * 2**(((note) - 10) / 12)
        sample = (np.sin(2 * np.pi * np.arange(self.rate * duration) * f / self.rate)).astype(np.float32)
        self.stream.write(self.volume * sample)

    def stop(self):
        self.stream.stop_stream()
        self.stream.close() 
        self.p.terminate()

    def __enter__(self):
        return self

    def __exit__(self, type, value, traceback):
        self.stop()

In [ ]:
with Tone() as tone: 
    for n in [1, 3, 5, 6, 8, 10, 12]:
        tone.play(3, n, 4)

In [ ]:
from pop import Tone

shoolBall1 = ((4, "SOL", 1/4), (4, "SOL", 1/4), (4, "RA", 1/4), (4, "RA", 1/4), 
              (4, "SOL", 1/4), (4, "SOL", 1/4), (4, "MI", 1/2), (4, "SOL", 1/4), 
              (4, "SOL", 1/4), (4, "MI", 1/4), (4, "MI", 1/4), (4, "RE", 1/2 + 1/4))

shoolBall2 = (*shoolBall1[:7], (4, "SOL", 1/4), (4, "MI", 1/2), (4, "SOL", 1/4), 
              (4, "MI", 1/4), (4, "RE", 1/4), (4, "MI", 1/4), (4, "DO", 1/2 + 1/4))

with Tone() as tone:
    tone.setTempo(200)

    for n in shoolBall1:
        tone.play(*n)
    tone.rest(1/4)

    for n in shoolBall2:
        tone.play(*n)
    tone.rest(1/4)

- 메뉴의 Kernel > Interrupt the Kernel 누르면 종료   

In [ ]:
import pyaudio
import wave

CHUNK = 1024 
RATE = 48000

p = pyaudio.PyAudio()
stream = p.open(format=pyaudio.paInt16,
                channels=1,
                rate=RATE,
                input=True,
                frames_per_buffer=CHUNK)

w = wave.open("./out_blocking.wav", "wb")
w.setnchannels(1)
w.setsampwidth(p.get_sample_size(pyaudio.paInt16))
w.setframerate(RATE)

try:
    while True:
        w.writeframes(stream.read(CHUNK))
except KeyboardInterrupt:
    pass

w.close()
stream.stop_stream()
stream.close()
p.terminate()

In [ ]:
# 녹음 시간 설정 변경된 코드
import pyaudio
import wave

CHUNK = 1024 
RATE = 48000

p = pyaudio.PyAudio()
stream = p.open(format=pyaudio.paInt16,
                channels=1,
                rate=RATE,
                input=True,
                frames_per_buffer=CHUNK)

w = wave.open("./out_blocking.wav", "wb")
w.setnchannels(1)
w.setsampwidth(p.get_sample_size(pyaudio.paInt16))
w.setframerate(RATE)

TIME = 5   # 5초
data = []

for _ in range(0, int(RATE / CHUNK * TIME)):
    d = stream.read(CHUNK)
    data.append(d)

w.writeframes(b''.join(data))

w.close()
stream.stop_stream()
stream.close()
p.terminate()

In [ ]:
import pyaudio
import wave
import time

CHUNK = 1024
RATE = 48000
isStop = False

In [ ]:
p = pyaudio.PyAudio()

w = wave.open("out_nonblocking.wav", 'wb')
w.setsampwidth(p.get_sample_size(pyaudio.paInt16))
w.setnchannels(1)
w.setframerate(RATE)

In [ ]:
def callback(in_data, frame_count, time_info, status):
    w.writeframes(in_data)
    data = chr(0) * len(in_data)
    return (data, pyaudio.paContinue if not isStop else pyaudio. paComplete)

In [ ]:
stream = p.open(format=pyaudio.paInt16, 
                channels=1, rate=RATE, input=True, 
                frames_per_buffer=CHUNK, 
                stream_callback=callback)

- 메뉴에서 Kernel > Interrupt the Kernel 클릭해 종료   

In [ ]:
stream.start_stream()
try:
    print("Recording..", end='')
    while True:
        print(".", end="")
        time.sleep(0.5)
except KeyboardInterrupt:
    isStop = True
    time.sleep(1)   # 녹음을 완료할 때까지 대기
    
w.close()
stream.stop_stream()
stream.close()
p.terminate()

In [ ]:
import pyaudio
import audioop 
import time

CHUNK = 1024
RATE = 48000
isStop = False

p = pyaudio.PyAudio()

# RMS 계산 및 소음 크기에 비례한 가로 막대 그래프를 그리는 사용자 콜백 메소드
def callback(in_data, frame_count, time_info, status):
    rms = audioop.rms(in_data, 2)
    print('=' * (rms//50), '*(', rms, ')')
    data = chr(0) * len(in_data)
    return (data, pyaudio.paContinue if not isStop else pyaudio.paAbort)

stream = p.open(format=pyaudio.paInt16,
                channels=1,
                rate=RATE,
                input=True,
                frames_per_buffer=CHUNK,
                stream_callback=callback)
stream.start_stream()

try:
    while True:
        time.sleep(0.1)
except KeyboardInterrupt:
    isStop = True

stream.stop_stream()
stream.close()
p.terminate()

In [ ]:
import time
from pop import *

sm = SoundMeter()

def onSoundMeter(rms, inData):
    if(rms>600):
        print(rms)

sm.setCallback(onSoundMeter)

input("input something")

sm.stop()

In [ ]:
from pop import AudioRecord, AudioPlay
import time

with AudioRecord("my_record.wav") as record:
    record.run()
    print("Start Recording...") 

    for _ in range(5):
        time.sleep(1)

    record.stop()
    print("Stop Recording...")

In [ ]:
with AudioPlay("my_record.wav", False, True) as play:   
    play.run()
    print("Start Play...")
    for _ in range(12):  
        time.sleep(1)

    play.stop()
    print("Stop play...")